In [14]:
using Healpix
using LinearAlgebra
using StaticArrays
using Base.Threads
using WignerD
using NPZ
using Plots
#using PyPlot
#using PyCall
using BenchmarkTools
using Random
#hp = pyimport("healpy")
include("../src/function2/rotator.jl")
include("../src/function2/tools.jl")

unique_theta_val (generic function with 1 method)

# Two time rotation

$$
D^{\ell}_{mn}(\phi + \delta \phi, \theta + \delta \theta, \psi + \delta \psi)
= \sum_{M} D^{\ell}_{mM}(\phi, \theta, 0) D^{\ell}_{Mn}(\alpha, \beta, \gamma)
$$

$$

R(\phi,\theta,\psi)=R_z(\phi)R_y(\theta)R_z(\psi),\qquad\\
R_2 = R(\phi,\theta,\psi)^{-1}\,R(\phi+\delta\phi,\theta+\delta\theta,\psi+\delta\psi) \\
\beta = \arccos\!\left((R_2)_{33}\right),\qquad \\
\alpha = \operatorname{atan2}\!\left((R_2)_{23},(R_2)_{13}\right),\qquad \\

\gamma = \operatorname{atan2}\!\left((R_2)_{32},-(R_2)_{31}\right)

$$

In [15]:
ell =  700
θ = pi/3
φ = pi/4
dθ = 1e-3
dφ = -1e-3
ψ = pi/6
err, (α2, β2, γ2) = check_split(φ, θ, dφ, dθ, ψ)

(5.911937606128959e-14, (-0.7141170019360827, 0.0013230392084807063, 1.2372162105158164))

In [16]:
D_tar = WignerD.wignerD(ell, φ + dφ, θ + dθ, ψ);

In [17]:
D_1st = WignerD.wignerD(ell, φ, θ, 0.0);

In [18]:
D_2nd = WignerD.wignerD(ell, α2, β2, γ2);

In [19]:
D_2time = D_1st * D_2nd ;

In [20]:
maximum(abs.(D_tar .- D_2time))

6.830101375729925e-12

# All-sky convolution
## How to calculate WignerD matrices

From All-sky convolution
There is algorithm to calculate WignerD

$$
D_{mn}^{\ell}(\phi, \theta, \psi) = \sum_{|M| \leq \ell} \left[ D^{\ell}_{mM}(\phi - \pi/2, -\pi/2, \theta)\times D^{\ell}_{Mn}(0, \pi/2, \psi + \pi/2) \right] \\
= \sum_{|M| \leq \ell} \left[ \exp(-im(\phi - \pi/2)) d^{\ell}_{mM}(-\pi/2)\exp(-iM\theta)\times d^{\ell}_{Mn}(\pi/2)\exp(-in(\psi + \pi/2)) \right] \\
= \sum_{|M| \leq \ell} \left[ \exp(-im(\phi - \pi/2)) d^{\ell}_{Mm}(\pi/2)\exp(-iM\theta)\times d^{\ell}_{Mn}(\pi/2)\exp(-in(\psi + \pi/2)) \right]
$$
convinient equation
$$
d^{\ell}_{mn}\left(\frac{\pi}{2}\right) = (-1)^{\ell-n}d_{-mn}^{\ell}\left(\frac{\pi}{2}\right)=(-1)^{\ell+m}d^{\ell}_{m-n}\left(\frac{\pi}{2}\right)=(-1)^{m-n}d_{-m-n}^{\ell}\left(\frac{\pi}{2}\right) \\
d_{m-n}^{\ell}\left( \frac{\pi}{2}\right) = (-1)^{\ell+m}d_{mn}^{\ell} \left( \frac{\pi}{2}\right) 
$$

# All-sky convolution
## How to calculate WignerD matrices

From All-sky convolution
There is algorithm to calculate WignerD

$$
d_{mn}^{\ell}(\theta) = D_{mn}^{\ell}(0, \theta, 0) = \sum_{|M| \leq \ell} \left[ D^{\ell}_{mM}(- \pi/2, -\pi/2, \theta)\times D^{\ell}_{Mn}(0, \pi/2,  \pi/2) \right] \\
= \sum_{|M| \leq \ell} \left[ \exp(-im(- \pi/2)) d^{\ell}_{mM}(-\pi/2)\exp(-iM\theta)\times d^{\ell}_{Mn}(\pi/2)\exp(-in( \pi/2)) \right] \\
= \sum_{|M| \leq \ell} \left[  d^{\ell}_{Mm}(\pi/2)\exp(-iM\theta)\exp(im(\pi/2)) \times d^{\ell}_{Mn}(\pi/2)\exp(-in(\pi/2)) \right]
$$
convinient equation
$$
d^{\ell}_{mn}\left(\frac{\pi}{2}\right) = (-1)^{\ell-n}d_{-mn}^{\ell}\left(\frac{\pi}{2}\right)=(-1)^{\ell+m}d^{\ell}_{m-n}\left(\frac{\pi}{2}\right)=(-1)^{m-n}d_{-m-n}^{\ell}\left(\frac{\pi}{2}\right) \\
d_{m-n}^{\ell}\left( \frac{\pi}{2}\right) = (-1)^{\ell+m}d_{mn}^{\ell} \left( \frac{\pi}{2}\right) \\
d^{\ell}_{mn}\left(-\theta \right) = d^{\ell}_{nm}\left( \theta \right)
$$

In [38]:
ell, phi, theta, psi = 1000, pi/3, pi/4, pi/6

(1000, 1.0471975511965976, 0.7853981633974483, 0.5235987755982988)

In [ ]:
d = @time WignerD.wignerd(ell,pi/2);

  51.620 s (4 allocations: 91.64 MiB)


In [40]:
phi, theta, psi = pi/3, pi/1, pi/6
D = @time  WignerD.wignerD(ell, phi, theta, psi)
Dp = @time  WignerD_calculator_fast(d, ell, phi, theta, psi);

 52.528606 seconds (4 allocations: 122.193 MiB, 0.08% gc time)
  7.889787 seconds (11 allocations: 122.284 MiB)


In [42]:
maximum(abs.(D .-Dp))

5.33114290741084e-13

In [44]:
L = 2*ell + 1
result = Matrix{ComplexF64}(undef, L, L);

In [54]:
tmpB = Matrix{ComplexF64}(undef, L, L)
w    = Vector{ComplexF64}(undef, L)
p    = Vector{ComplexF64}(undef, L)
q    = Vector{ComplexF64}(undef, L)
;

In [53]:
@time WignerD_calculator!(result, d, ell, phi, theta, psi, tmpB=tmpB, w=w, p=p, q=q);

  8.504110 seconds (3 allocations: 112 bytes)


In [55]:
D = @time  WignerD.wignerD(ell, phi, theta, psi)

 56.896079 seconds (4 allocations: 122.193 MiB)


2001×2001 Matrix{ComplexF64}:
  2.95865e-14+1.97239e-27im  …          -0.5+0.866025im
  6.62264e-18-1.14707e-17im      1.53616e-15+2.6607e-15im
  7.53047e-16+1.30432e-15im      2.29445e-14-1.93372e-27im
 -3.42292e-18+9.02332e-31im      3.03149e-17-5.25069e-17im
  9.30692e-15-1.61201e-14im      8.79398e-15+1.52316e-14im
  4.75391e-17+8.23401e-17im  …   4.46146e-17-8.35115e-30im
 -1.45399e-14+2.02318e-27im       9.7415e-15-1.68728e-14im
  1.95904e-17-3.39316e-17im     -4.85111e-17-8.40237e-17im
  3.01551e-15+5.22302e-15im      -2.5201e-14+1.58052e-27im
  2.78083e-17-4.08217e-31im     -3.96013e-17+6.85915e-17im
  3.01275e-16-5.21824e-16im  …   1.45023e-14+2.51188e-14im
 -1.86821e-17-3.23584e-17im      5.92872e-18-9.81933e-31im
  1.47691e-15-5.09476e-28im       1.4761e-14-2.55668e-14im
             ⋮               ⋱              ⋮
 -5.92872e-18-9.81933e-31im      1.86821e-17-3.23584e-17im
  1.45023e-14-2.51188e-14im  …   3.01275e-16+5.21824e-16im
  3.96013e-17+6.85915e-17im     -2.78083e-1

In [24]:
nside =128

128

In [12]:
unique_theta_val(nside);

In [13]:
unique_theta_num(1000, nside)

(-278703, -280656)

In [23]:
0.006378890353433737 + 3.1352137632363597

3.1415926535897936

In [24]:
0.012757845597670932 + 3.1288348079921224

3.1415926535897936

In [127]:
nside = 4
npix = nside2npix(nside)
res = Resolution(nside)

Healpix resolution(NSIDE = 4)

In [129]:
num = 5000
Random.seed!(10)
phi = rand(num)*2pi
theta = rand(num)*pi
psi = rand(num)*2pi;

In [131]:
pix = zeros(Int64, 5000);

In [151]:
for phi_loop in enumerate(phi)
    idx = phi_loop[1]
    pix[idx] = ang2pixRing(res, theta[idx] ,phi[idx])
end

In [147]:
ang2pixRing(res, theta[1] ,phi[1])

192

In [153]:
pix

5000-element Vector{Int64}:
 192
  36
 191
 125
  58
   6
 131
 189
   4
 164
  69
  76
   2
   ⋮
  73
  18
 172
   2
 158
 191
 189
 137
  16
 189
   1
  66